# 1. Bronze Layer: Data Ingestion

This notebook ingests raw IoT sensor data from the landing zone (Unity Catalog Volume) into a Bronze Delta table.

The Bronze layer stores raw, unprocessed data, preserving all data quality issues for downstream processing.

## 2. Data Source

- Input: JSON files from Volume (landing zone)
- Path: `/Volumes/main/default/machine_health_volume/`
- Data characteristics:
  - Contains missing values
  - Contains inconsistent formats
  - Contains outliers

No transformations are applied at this stage.

In [0]:
# Read raw JSON data from the landing zone (Volume)
df = spark.read.json("/Volumes/main/default/machine_health_volume/") 
display(df.limit(10))

machine_id,pressure,temperature,timestamp,vibration
M2,4.65,74,2026-04-12T10:52:22.067473,0.88
M1,2.48,75,2026-04-12T10:52:22.067481,0.51
M1,1.54,90,2026-04-12T10:52:22.067486,5.0
M4,1.49,81,2026-04-12T10:52:22.067490,0.98
M4,4.89,null,2026-04-12T10:52:22.067494,0.83
M2,2.36,79,2026-04-12T10:52:22.067498,0.43
M5,4.26,null,2026-04-12T10:52:22.067502,0.65
M1,4.69,999,2026-04-12T10:52:22.067506,0.45
M5,4.32,66,2026-04-12T10:52:22.067510,0.42
M3,1.67,null,2026-04-12T10:52:22.067513,5.0


## 3. Ingestion Strategy

- Schema is inferred to preserve raw data
- No cleaning or casting is performed
- This ensures full traceability of source data

In [0]:
# Write raw data into Bronze Delta table
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("main.default.bronze_machine_readings")

## 4. Verification

Query the Bronze table to confirm successful ingestion.

In [0]:
df = spark.sql("""
SELECT *
FROM main.default.bronze_machine_readings
LIMIT 10
""")

display(df)

machine_id,pressure,temperature,timestamp,vibration
M3,3.45,85,2026-04-12T10:52:22.060563,0.75
M3,4.15,77,2026-04-12T10:52:22.060591,0.12
M1,3.22,78,2026-04-12T10:52:22.060599,0.85
M3,3.17,67,2026-04-12T10:52:22.060613,0.34
M5,3.58,67,2026-04-12T10:52:22.060619,0.29
M5,3.65,63,2026-04-12T10:52:22.060625,0.53
M5,4.2,null,2026-04-12T10:52:22.060630,5.0
M2,3.89,86,2026-04-12T10:52:22.060640,0.45
M1,2.4,73,2026-04-12T10:52:22.060646,0.72
M4,1.17,75,2026-04-12T10:52:22.060652,0.51
